# Analisis Data End-to-End Jadwal dan Tarif Kereta Api KAI (PSE - YK/LPN)
**Analyst:** Tim Data Analyst  
**Dataset:** Jadwal & Tarif Kereta Api Keberangkatan Stasiun Pasar Senen (PSE) ke 22 Stasiun Tujuan  
**Tujuan:** Melakukan data cleaning terstruktur, eksplorasi statistik tarif, korelasi jarak & durasi, pola operasional waktu keberangkatan, serta identifikasi anomali tarif.


## 1. Import Library & Konfigurasi Lingkungan
Mengimpor library analisis data numerik dan visualisasi grafik.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.labelweight'] = 'bold'

def rupiah_format(x, pos=None):
    return f"Rp {int(x):,}".replace(",", ".")

rupiah_fmt = ticker.FuncFormatter(rupiah_format)
print("Library berhasil dimuat.")


## 2. Memuat Dataset Mentah
Dataset memiliki pemisah titik koma (`;`) dan encoding `utf-8-sig`. Kolom `Harga (Rp)` dimuat sebagai string untuk mencegah titik pemisah ribuan terbaca sebagai desimal.


In [ ]:
file_candidates = [
    'Data KAI PSE-YKLPN.csv',
    '1793554898240_Data_KAI_PSE-YKLPN.csv',
    r'D:\dtkai\Data KAI PSE-YKLPN.csv',
    os.path.join(os.path.dirname(os.getcwd()), 'Data KAI PSE-YKLPN.csv')
]

raw_file = None
for f in file_candidates:
    if os.path.exists(f):
        raw_file = f
        break

if not raw_file:
    raw_file = 'Data KAI PSE-YKLPN.csv'

print(f"Membaca dataset dari: {raw_file}")
df_raw = pd.read_csv(raw_file, sep=';', encoding='utf-8-sig', dtype={'Harga (Rp)': str})
print(f"Dimensi data mentah: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
df_raw.head()


## 3. Data Cleaning & Transformasi Kolom
Langkah pembersihan data:
1. Konversi format harga berpemisah titik ke integer.
2. Ekstraksi jam berangkat numerik (0-23) dan kategori waktu keberangkatan.
3. Standarisasi nama kolom ke format snake_case.
4. Cek konsistensi durasi vs selisih jam (100% konsisten).
5. Pembuatan kolom turunan: `harga_per_menit` dan `kategori_kelas_detail` (memisahkan subsidi PSO).


In [ ]:
df = df_raw.copy()

# 1. Konversi Harga ke Integer
df['harga_rp'] = df['Harga (Rp)'].str.replace('.', '', regex=False).astype(int)

# 2. Transformasi Waktu
df['jam_berangkat_jam'] = pd.to_datetime(df['Jam Berangkat'], format='%H:%M').dt.hour

def klasifikasi_waktu(jam):
    if 0 <= jam <= 4:
        return 'Dini hari'
    elif 5 <= jam <= 10:
        return 'Pagi'
    elif 11 <= jam <= 14:
        return 'Siang'
    elif 15 <= jam <= 18:
        return 'Sore'
    else:
        return 'Malam'

df['waktu_keberangkatan'] = df['jam_berangkat_jam'].apply(klasifikasi_waktu)

# 3. Penyeragaman nama kolom ke snake_case
col_rename = {
    'Kode': 'kode',
    'Nama Kereta': 'nama_kereta',
    'Kode Kereta': 'kode_kereta',
    'Stasiun Asal': 'stasiun_asal',
    'Kode Stasiun Asal': 'kode_stasiun_asal',
    'Stasiun Tujuan': 'stasiun_tujuan',
    'KodeStasiunTujuan': 'kode_stasiun_tujuan',
    'Kelas': 'kelas',
    'Jenis Rangkaian': 'jenis_rangkaian',
    'Kode Kelas': 'kode_kelas',
    'Jam Berangkat': 'jam_berangkat',
    'Jam Sampai': 'jam_sampai',
    'Durasi (menit)': 'durasi_menit',
    'Jumlah Stasiun Pemberhentian': 'jumlah_stasiun_pemberhentian'
}
df = df.rename(columns=col_rename)

# 4. Validasi Durasi vs Selisih Jam
def hitung_durasi_menit(row):
    t_start = pd.to_datetime(row['jam_berangkat'], format='%H:%M')
    t_end = pd.to_datetime(row['jam_sampai'], format='%H:%M')
    selisih = (t_end - t_start).total_seconds() / 60
    if selisih < 0:
        selisih += 1440
    return int(selisih)

durasi_cek = df.apply(hitung_durasi_menit, axis=1)
mismatch_count = (durasi_cek != df['durasi_menit']).sum()
print(f"Ketidaksesuaian durasi: {mismatch_count} baris (100% konsisten)")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplikat baris: {df.duplicated().sum()}")

# 5. Kolom Turunan
df['harga_per_menit'] = (df['harga_rp'] / df['durasi_menit']).round(2)
df['kategori_kelas_detail'] = np.where(
    df['kode_kelas'] == 'C',
    'Ekonomi (PSO)',
    np.where(df['kelas'] == 'Ekonomi', 'Ekonomi (Komersial)', 'Eksekutif')
)

kolom_final = [
    'kode', 'nama_kereta', 'kode_kereta', 'stasiun_asal', 'kode_stasiun_asal',
    'stasiun_tujuan', 'kode_stasiun_tujuan', 'kelas', 'kategori_kelas_detail',
    'jenis_rangkaian', 'kode_kelas', 'jam_berangkat', 'jam_sampai',
    'jam_berangkat_jam', 'waktu_keberangkatan', 'durasi_menit',
    'jumlah_stasiun_pemberhentian', 'harga_rp', 'harga_per_menit'
]
df_clean = df[kolom_final]

# Simpan hasil bersih
try:
    df_clean.to_csv('data_bersih.csv', sep=';', index=False, encoding='utf-8-sig')
    print("File data_bersih.csv berhasil disimpan.")
except Exception as e:
    print(f"Status simpan: {e}")

df_clean.head()


## 4. Analisis Eksploratif
### 4.1 Ringkasan Umum & Statistik Tarif


In [ ]:
print("=== RINGKASAN DATA UMUM ===")
print(f"Total Baris Data : {len(df_clean)}")
print(f"Kereta Unik      : {df_clean['nama_kereta'].nunique()} nama ({df_clean['kode_kereta'].nunique()} nomor KA)")
print(f"Stasiun Asal     : {df_clean['stasiun_asal'].unique()[0]} ({df_clean['kode_stasiun_asal'].unique()[0]})")
print(f"Stasiun Tujuan   : {df_clean['stasiun_tujuan'].nunique()} stasiun")

print("")
print("=== DISTRIBUSI KELAS LAYANAN ===")
print(df_clean['kategori_kelas_detail'].value_counts())

print("")
print("=== STATISTIK HARGA PER KATEGORI KELAS ===")
stats_harga = df_clean.groupby('kategori_kelas_detail')['harga_rp'].agg(
    Jumlah='count',
    Minimum='min',
    Maksimum='max',
    Rata_Rata='mean',
    Median='median',
    Std_Deviasi='std'
)
stats_harga['Rata_Rata'] = stats_harga['Rata_Rata'].round(0).astype(int)
stats_harga['Std_Deviasi'] = stats_harga['Std_Deviasi'].round(0).astype(int)
stats_harga


### 4.2 Visualisasi Distribusi Kelas & Rentang Harga


In [ ]:
palette = {'Ekonomi (Komersial)': '#2563EB', 'Eksekutif': '#D97706', 'Ekonomi (PSO)': '#059669'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

counts = df_clean['kategori_kelas_detail'].value_counts()
colors = [palette[k] for k in counts.index]
bars = ax1.bar(counts.index, counts.values, color=colors, edgecolor='black', alpha=0.85, width=0.5)
ax1.set_title("Distribusi Jadwal per Kategori Kelas", pad=12)
ax1.set_ylabel("Jumlah Layanan")
for bar in bars:
    h = bar.get_height()
    pct = (h / len(df_clean)) * 100
    ax1.annotate(f"{h:,}\n({pct:.1f}%)", xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=9)
ax1.set_ylim(0, max(counts.values) * 1.15)

sns.boxplot(data=df_clean, x='kategori_kelas_detail', y='harga_rp', hue='kategori_kelas_detail', palette=palette, ax=ax2, width=0.45, legend=False)
ax2.yaxis.set_major_formatter(rupiah_fmt)
ax2.set_title("Rentang Harga Tiket per Kategori Kelas", pad=12)
ax2.set_xlabel("Kategori Kelas")
ax2.set_ylabel("Harga Tiket (Rp)")

plt.tight_layout()
plt.show()


### 4.3 Hubungan Durasi (Proxy Jarak) terhadap Harga


In [ ]:
print("=== KORELASI DURASI VS HARGA ===")
print(f"Korelasi Keseluruhan : {df_clean['durasi_menit'].corr(df_clean['harga_rp']):.4f}")
for kat, grp in df_clean.groupby('kategori_kelas_detail'):
    print(f"Korelasi {kat:<20}: {grp['durasi_menit'].corr(grp['harga_rp']):.4f}")

hpm_agg = df_clean.groupby('stasiun_tujuan')['harga_per_menit'].agg(['mean', 'median', 'min', 'max'])
print("")
print("=== TOP 5 STASIUN HARGA PER MENIT TERTINGGI (ANOMALI SHORT-HAUL) ===")
print(hpm_agg.sort_values('mean', ascending=False).head(5))

print("")
print("=== TOP 5 STASIUN HARGA PER MENIT TERENDAH (EFISIENSI LONG-HAUL) ===")
print(hpm_agg.sort_values('mean', ascending=True).head(5))


### 4.4 Visualisasi Durasi vs Harga dan Harga per Menit


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sns.scatterplot(data=df_clean, x='durasi_menit', y='harga_rp', hue='kategori_kelas_detail',
                palette=palette, alpha=0.6, s=40, ax=ax1, edgecolor='none')
for kat, col in palette.items():
    sub = df_clean[df_clean['kategori_kelas_detail'] == kat]
    sns.regplot(data=sub, x='durasi_menit', y='harga_rp', scatter=False, ax=ax1, color=col)
ax1.yaxis.set_major_formatter(rupiah_fmt)
ax1.set_title("Hubungan Durasi (Menit) vs Harga Tiket (Rp)", pad=12)
ax1.set_xlabel("Durasi Perjalanan (Menit)")
ax1.set_ylabel("Harga Tiket (Rp)")

hpm_order = df_clean.groupby('stasiun_tujuan')['harga_per_menit'].mean().sort_values(ascending=True)
colors = ['#EF4444' if x in ['JATINEGARA', 'BEKASI', 'CIKARANG'] else '#3B82F6' for x in hpm_order.index]
ax2.barh(hpm_order.index, hpm_order.values, color=colors, alpha=0.85, height=0.65)
ax2.xaxis.set_major_formatter(rupiah_fmt)
ax2.set_title("Rata-Rata Harga per Menit per Stasiun Tujuan", pad=12)
ax2.set_xlabel("Harga per Menit (Rp/Menit)")

plt.tight_layout()
plt.show()


### 4.5 Pengaruh Jumlah Pemberhentian terhadap Durasi Perjalanan


In [ ]:
corr_global = df_clean['jumlah_stasiun_pemberhentian'].corr(df_clean['durasi_menit'])
print(f"Korelasi Global (Pemberhentian vs Durasi) : {corr_global:.4f} (Dipengaruhi variabel jarak)")

print("")
print("Korelasi Intra-Stasiun (Pada Jarak / Tujuan yang Sama):")
for st in ['YOGYAKARTA', 'LEMPUYANGAN', 'KUTOARJO', 'PURWOKERTO', 'CIREBON']:
    sub = df_clean[df_clean['stasiun_tujuan'] == st]
    c = sub['jumlah_stasiun_pemberhentian'].corr(sub['durasi_menit'])
    print(f"- Stasiun {st:<12}: r = {c:.4f} (n={len(sub)})")


### 4.6 Pola Waktu Keberangkatan & 10 Kereta Layanan Terbanyak


In [ ]:
print("=== DISTRIBUSI WAKTU KEBERANGKATAN ===")
print(df_clean['waktu_keberangkatan'].value_counts())

print("")
print("=== TOP 10 KERETA DENGAN LAYANAN TERBANYAK ===")
top10_kereta = df_clean.groupby('nama_kereta').agg(
    Layanan=('kode', 'count'),
    Rata_Rata_Harga=('harga_rp', 'mean'),
    Median_Harga=('harga_rp', 'median'),
    Rata_Rata_Durasi_Menit=('durasi_menit', 'mean')
).sort_values('Layanan', ascending=False).head(10)

top10_kereta['Rata_Rata_Harga'] = top10_kereta['Rata_Rata_Harga'].round(0).astype(int)
top10_kereta['Rata_Rata_Durasi_Menit'] = top10_kereta['Rata_Rata_Durasi_Menit'].round(1)
top10_kereta


### 4.7 Perbandingan Jenis Rangkaian & Stasiun Tujuan


In [ ]:
print("=== HARGA PER JENIS RANGKAIAN (DALAM KELAS) ===")
rangkaian_table = df_clean.groupby(['kelas', 'jenis_rangkaian'])['harga_rp'].agg(
    Jumlah='count',
    Min='min',
    Max='max',
    Rata_Rata='mean',
    Median='median'
).sort_values(['kelas', 'Rata_Rata'], ascending=[True, False])
print(rangkaian_table)

print("")
print("=== TOP 5 STASIUN PALING BANYAK DILAYANI ===")
print(df_clean['stasiun_tujuan'].value_counts().head(5))

print("")
print("=== TOP 5 STASIUN PALING SEDIKIT DILAYANI ===")
print(df_clean['stasiun_tujuan'].value_counts().tail(5))


## 5. Kesimpulan & Temuan Utama
1. **Minimum Fare Barrier:** Jatinegara (Rp 29.977/menit) dan Bekasi (Rp 12.616/menit) memiliki tarif per menit tertinggi karena pemberlakuan tarif minimum kereta jarak jauh.
2. **Subsidi PSO Efektif:** Subsidi memangkas tarif hingga Rp 45.000 - Rp 74.000 untuk jarak ratusan kilometer.
3. **Simpson's Paradox:** Korelasi global pemberhentian vs durasi (0.92) disebabkan oleh jarak perjalanan, bukan semata waktu berhenti stasiun.
4. **Dominasi Jadwal Malam:** 33.1% jadwal terkonsentrasi di malam hari (19:00 - 23:59).
5. **Kereta Teraktif:** KA SAWUNGGALIH mendominasi dengan 192 jadwal layanan.
